# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

Submit "ratings.tsv" in Gradescope.

In [3]:
import pandas as pd

# Load the recipes and tags datasets
recipes_df = pd.read_csv('/content/dx704-project-04/recipes.tsv', sep='\t')
tags_df = pd.read_csv('/content/dx704-project-04/recipe-tags.tsv', sep='\t')

print("Recipes:", recipes_df.shape)
print("Tags:", tags_df.shape)
display(recipes_df[["recipe_slug", "recipe_title"]].head())

Recipes: (100, 3)
Tags: (752, 2)


,recipe_slug,recipe_title
0,falafel,Falafel
1,spamburger,Spamburger
2,bacon-fried-rice,Bacon Fried Rice
3,chicken-fingers,Chicken Fingers
4,apple-crisp,Apple Crisp


In [4]:
def raw_rating(title):
    title = title.lower()

    # This persona dislikes Spam and bacon-heavy dishes.
    if any(word in title for word in ("bacon", "spam")):
        return 1

    # Favorite Latin and Mediterranean dishes, plus a few sweets.
    if any(word in title for word in (
        "falafel", "pasta", "lasagna", "tacos", "enchiladas",
        "tamales", "burritos", "quesadillas", "fajitas",
        "guacamole", "pico de gallo", "chiles rellenos",
        "brownies", "chocolate cake", "chocolate babka"
    )):
        return 5

    if any(word in title for word in (
        "pickled", "cranberry", "fried oysters"
    )):
        return 2

    # Dishes and occasional desserts this persona likes.
    if any(word in title for word in (
        "chicken", "chocolate", "croissant", "nacho", "noodle",
        "bolognese", "quiche", "rice", "beef", "ramen", "udon",
        "crisp", "crumble", "cobbler", "pie", "cupcake",
        "babka", "danish", "souffle"
    )):
        return 4

    return 3

In [5]:
ratings = recipes_df[["recipe_slug"]].copy()

# Assign raw 1–5 ratings, then convert to the required 0–1 scale.
ratings["rating_raw"] = recipes_df["recipe_title"].map(raw_rating)
ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25

num_zeros = (ratings["rating"] == 0).sum()
num_ones = (ratings["rating"] == 1).sum()

assert len(ratings) == 100
assert ratings["recipe_slug"].is_unique
assert ratings["rating"].between(0, 1).all()
assert num_zeros >= 10, f"Need at least 10 zeros; found {num_zeros}"
assert num_ones >= 10, f"Need at least 10 ones; found {num_ones}"

output_path = "/content/dx704-project-04/ratings.tsv"
ratings[["recipe_slug", "rating"]].to_csv(
    output_path, sep="\t", index=False
)

print(f"Saved {output_path}")
print(f"Ratings of 0: {num_zeros}; ratings of 1: {num_ones}")
display(ratings.head(10))

Saved /content/dx704-project-04/ratings.tsv
Ratings of 0: 15; ratings of 1: 22


,recipe_slug,rating_raw,rating
0,falafel,5,1.00
1,spamburger,1,0.00
2,bacon-fried-rice,1,0.00
3,chicken-fingers,4,0.75
4,apple-crisp,4,0.75
5,cranberry-apple-crisp,2,0.25
6,bacon-chocolate-chip-cookies,1,0.00
7,sujebi,3,0.50
8,pasta-primavera,5,1.00
9,ramen,4,0.75


In [6]:
ratings[["recipe_slug", "rating"]].to_csv(
    "/content/dx704-project-04/ratings.tsv",
    sep="\t",
    index=False
)

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [14]:
# YOUR CHANGES HERE

# Create one 0/1 column for every recipe tag.
tag_features = pd.crosstab(
    tags_df["recipe_slug"],
    tags_df["recipe_tag"]
).clip(upper=1)

# Match the row order in ratings.tsv.
tag_features = tag_features.reindex(
    ratings["recipe_slug"], fill_value=0
).astype(int)

features = tag_features.reset_index()
features.insert(1, "bias", 1)

# Check the required structure and order.
assert features["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()
assert (features["bias"] == 1).all()
assert features.iloc[:, 2:].isin([0, 1]).all().all()

features.to_csv(
    "/content/dx704-project-04/features.tsv",
    sep="\t",
    index=False
)

print("Saved features.tsv")
print("Shape:", features.shape)
display(features.iloc[:5, :10])

Saved features.tsv
Shape: (100, 298)


recipe_tag,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus
0,falafel,1,0,0,0,1,0,0,0,0
1,spamburger,1,0,0,0,0,0,0,0,0
2,bacon-fried-rice,1,0,0,0,0,0,0,0,0
3,chicken-fingers,1,0,0,0,1,0,0,0,0
4,apple-crisp,1,0,0,0,0,0,1,0,0


Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [15]:
# YOUR CHANGES HERE

from sklearn.linear_model import Ridge

# The bias column is already in features, so no separate intercept is needed.
feature_columns = features.columns.drop("recipe_slug")
X = features[feature_columns].to_numpy(dtype=float)
y = ratings["rating"].to_numpy(dtype=float)

assert features["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()

model = Ridge(alpha=1, fit_intercept=False)
model.fit(X, y)

coefficients = pd.DataFrame({
    "recipe_tag": feature_columns,
    "coefficient": model.coef_
})



Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [16]:
# YOUR CHANGES HERE

coefficients.to_csv(
    "/content/dx704-project-04/model.tsv",
    sep="\t",
    index=False
)

print("Saved model.tsv with", len(coefficients), "coefficients")
display(coefficients.head(10))

Saved model.tsv with 297 coefficients


,recipe_tag,coefficient
0,bias,0.495152
1,alfredo,0.054681
2,almond,0.026438
3,american,0.011642
4,appetizer,-0.039245
5,appetizers,0.017049
6,apple,-0.013317
7,asiancuisine,0.016728
8,asparagus,-0.022995
9,avocado,-0.043855


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [17]:
# YOUR CHANGES HERE

estimates = pd.DataFrame({
    "recipe_slug": features["recipe_slug"],
    "score_estimate": model.predict(X)
})

assert len(estimates) == 100
assert estimates["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()

estimates.to_csv(
    "/content/dx704-project-04/estimates.tsv",
    sep="\t",
    index=False
)

print("Saved estimates.tsv")
display(estimates.head(10))

Saved estimates.tsv


,recipe_slug,score_estimate
0,falafel,0.912167
1,spamburger,0.052298
2,bacon-fried-rice,0.034712
3,chicken-fingers,0.698044
4,apple-crisp,0.715029
5,cranberry-apple-crisp,0.317376
6,bacon-chocolate-chip-cookies,0.083207
7,sujebi,0.521000
8,pasta-primavera,0.923592
9,ramen,0.751704


Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [18]:
# YOUR CHANGES HERE

import numpy as np

linucb_alpha = 2.0

# One observation per recipe; ridge regularization is 1.
A = np.eye(X.shape[1]) + X.T @ X

# For each recipe x, uncertainty = sqrt(xᵀ A⁻¹ x).
A_inverse_X_transpose = np.linalg.solve(A, X.T)
variances = np.sum(X * A_inverse_X_transpose.T, axis=1)
uncertainty = np.sqrt(np.maximum(variances, 0))

bounds = pd.DataFrame({
    "recipe_slug": features["recipe_slug"],
    "score_bound": estimates["score_estimate"]
                   + linucb_alpha * uncertainty
})

assert len(bounds) == 100
assert (bounds["score_bound"] >= estimates["score_estimate"]).all()

bounds.to_csv(
    "/content/dx704-project-04/bounds.tsv",
    sep="\t",
    index=False
)

print("Saved bounds.tsv")
display(bounds.head(10))

Saved bounds.tsv


,recipe_slug,score_bound
0,falafel,2.678422
1,spamburger,1.946634
2,bacon-fried-rice,1.900869
3,chicken-fingers,2.492160
4,apple-crisp,2.491371
5,cranberry-apple-crisp,2.143501
6,bacon-chocolate-chip-cookies,1.771363
7,sujebi,2.345895
8,pasta-primavera,2.608627
9,ramen,2.662699


Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [19]:
# YOUR CHANGES HERE
# Start with no observations.
A_online = np.eye(X.shape[1])
b_online = np.zeros(X.shape[1])

recommendation_rows = []

for step in range(100):
    # Current coefficient estimates.
    theta = np.linalg.solve(A_online, b_online)

    # Calculate an upper bound for every recipe.
    A_inverse_X_transpose = np.linalg.solve(A_online, X.T)
    variances = np.sum(X * A_inverse_X_transpose.T, axis=1)
    score_bounds = (
        X @ theta
        + linucb_alpha * np.sqrt(np.maximum(variances, 0))
    )

    # Recommend the recipe with the highest bound.
    chosen = int(np.argmax(score_bounds))
    recipe_slug = features["recipe_slug"].iloc[chosen]
    reward = float(y[chosen])

    # Record the bound BEFORE updating with this reward.
    recommendation_rows.append(
        (recipe_slug, float(score_bounds[chosen]), reward)
    )

    # Learn from the recommendation.
    x = X[chosen]
    A_online += np.outer(x, x)
    b_online += reward * x

recommendations = pd.DataFrame(
    recommendation_rows,
    columns=["recipe_slug", "score_bound", "reward"]
)

assert len(recommendations) == 100

recommendations.to_csv(
    "/content/dx704-project-04/recommendations.tsv",
    sep="\t",
    index=False
)

print("Saved recommendations.tsv")
print("Total reward:", recommendations["reward"].sum())
display(recommendations.head(10))

Saved recommendations.tsv
Total reward: 65.25


,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,0.75
1,ma-la-chicken,7.242589,0.75
2,quesadillas,7.264733,1.00
3,ramen,7.290383,0.75
4,spamburger,7.046758,0.00
5,pain-au-chocolat,7.004756,0.50
6,bacon-fried-rice,7.000624,0.00
7,chocolate-babka,6.896560,1.00
8,nacho-fries,6.875644,0.75
9,cranberry-sauce,6.721225,0.25


Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.